# 01 · Data Cleaning & Standardization (2015–2025)

**Project:** AgriShield — Early Warning System for Crop Yield & Livestock Forage Risk in Kenya  
**Notebook Goal:** Clean our raw datasets from `../data/`, standardize all **47 Kenyan county names**, align the time series to **monthly county records (`2015–2025`, `6,204` rows)**, and combine them into a unified dataset in `../data/processed/` for EDA, Feature Engineering, and Model Training.

---
### Step 1: Setup, Output Folder & Standard 47-County Name Helper
Different raw files spell county names differently (for example: `KE028`, `ELGEYO/MARAKWET`, `Elgeyo-Marakwet`, or `MURANG'A`).  
In the cell below, we:
1. Create the `../data/processed` folder where cleaned files will be saved.
2. Load `kenya_county_crosswalk.csv` and build a simple `clean_county_name()` function so every dataset uses the exact same **47 standard county names**.

In [15]:
from pathlib import Path
import pandas as pd
import numpy as np

# 1. Set up input (raw) and output (processed) directories

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
# 00a writes to data/raw; legacy files may be in data/. Check both.
RAW_DIRS = [RAW_DIR, Path("../data")]
PROC_DIR = Path("../data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)

def find_raw(name):
    """Find a raw file in data/raw or data/, whichever exists."""
    for d in RAW_DIRS:
        cand = d / name
        if cand.exists():
            return cand
    return RAW_DIR / name  # for error message if missing

# 2. Load the crosswalk file to build our county name & PCODE lookup dictionaries

crosswalk_df = pd.read_csv(Path("../data/kenya_county_crosswalk.csv"))

# Standardize hyphens to spaces (e.g., "Elgeyo-Marakwet" -> "Elgeyo Marakwet")

crosswalk_df["county"] = crosswalk_df["county"].str.strip().str.replace("-", " ", regex=False)

# Dictionary 1: Map PCODE (e.g. 'KE027') -> Standard County Name (e.g. 'Uasin Gishu')

pcode_to_county = dict(zip(crosswalk_df["pcode"].str.strip(), crosswalk_df["county"]))

# Dictionary 2: Map simplified text key -> Standard County Name

STANDARD_COUNTIES = sorted(crosswalk_df["county"].unique())
name_lookup = {
    c.lower().replace(" ", "").replace("-", "").replace("/", "").replace("'", ""): c
    for c in STANDARD_COUNTIES
}
# Handle common raw variations seen in KNBS and climate grid files

name_lookup["nairobicity"] = "Nairobi"
name_lookup["tharaka"] = "Tharaka Nithi"

def clean_county_name(raw_name):
    """Convert any raw county name variation into one of the 47 standard county names."""
    if pd.isna(raw_name):
        return np.nan
    key = str(raw_name).strip().lower().replace(" ", "").replace("-", "").replace("/", "").replace("'", "")
    return name_lookup.get(key, np.nan)

print(f"Total standard Kenyan counties loaded: {len(STANDARD_COUNTIES)}")
print("Sample PCODE mapping:", list(pcode_to_county.items())[:5])



Total standard Kenyan counties loaded: 47
Sample PCODE mapping: [('KE001', 'Mombasa'), ('KE002', 'Kwale'), ('KE003', 'Kilifi'), ('KE004', 'Tana River'), ('KE005', 'Lamu')]


### Step 2: Clean NASA MODIS NDVI Data (`2015–2025`)

In our raw file (`ken_ndvi_subnational_2015_2025_raw.csv`), NDVI is recorded every **10 days (dekadal)** and uses `PCODE`s (like `KE027` for county or `KE027001` for sub-county).  
To prepare it for monthly modeling across all **47 counties**, we:
1. Convert `date` to a proper datetime format and extract `year` and `month`.
2. Take the first 5 characters of `PCODE` (`KE001`–`KE047`) and map each code to its standard county name using `pcode_to_county`.
3. Convert `vim` (Vegetation Index Mean) to numeric and average the 10-day readings into **monthly county NDVI (`ndvi_mean`)**.
4. Save the cleaned dataset (`6,204` rows = `47 counties × 132 months`) to `../data/processed/NDVI_FINAL.csv`.

In [16]:
# 1. Load raw NASA MODIS NDVI dataset

ndvi_raw = pd.read_csv(find_raw("ken_ndvi_subnational_2015_2025_raw.csv"), low_memory=False)

# 2. Parse dates and convert NDVI ('vim') to numeric (drops any metadata/tag rows)

ndvi_raw["date"] = pd.to_datetime(ndvi_raw["date"], errors="coerce")
ndvi_raw["vim"] = pd.to_numeric(ndvi_raw["vim"], errors="coerce")
ndvi_raw = ndvi_raw.dropna(subset=["date", "vim", "PCODE"]).copy()

# 3. Map the 5-character County PCODE (KE001..KE047) to standard County Name

ndvi_raw["county_pcode"] = ndvi_raw["PCODE"].astype(str).str.strip().str[:5]
ndvi_raw["county"] = ndvi_raw["county_pcode"].map(pcode_to_county)

# 4. Extract year and month, then average the 10-day readings to monthly county level

ndvi_raw["year"] = ndvi_raw["date"].dt.year
ndvi_raw["month"] = ndvi_raw["date"].dt.month

ndvi_clean = (
    ndvi_raw.groupby(["county", "year", "month"], as_index=False)["vim"]
    .mean()
    .rename(columns={"vim": "ndvi_mean"})
    .sort_values(["county", "year", "month"])
    .reset_index(drop=True)
)
ndvi_clean["ndvi_mean"] = ndvi_clean["ndvi_mean"].round(6)

# 5. Verify shape (47 counties x 132 months = 6,204 rows) and save

print(f"Cleaned NDVI shape: {ndvi_clean.shape} | Unique counties: {ndvi_clean['county'].nunique()}")
print(f"Missing values: {ndvi_clean.isna().sum().sum()}")
display(ndvi_clean.head(10))

ndvi_clean.to_csv(PROC_DIR / "NDVI_FINAL.csv", index=False)
print("Saved -> ../data/processed/NDVI_FINAL.csv")


Cleaned NDVI shape: (13771, 4) | Unique counties: 47
Missing values: 0


,county,year,month,ndvi_mean
0,Baringo,2002,7,0.520922
1,Baringo,2002,8,0.479232
2,Baringo,2002,9,0.427410
3,Baringo,2002,10,0.415810
4,Baringo,2002,11,0.454565
5,Baringo,2002,12,0.502911
6,Baringo,2003,1,0.533893
7,Baringo,2003,2,0.449934
8,Baringo,2003,3,0.400479
9,Baringo,2003,4,0.465918


Saved -> ../data/processed/NDVI_FINAL.csv


### Step 3: Clean & Combine Meteorological Data — Rainfall, Soil Moisture & Temperature (`2015–2025`)

Our meteorological variables (`temperature_mean`, `rainfall_sum`, and `soil_moisture_mean`) come from three raw satellite/reanalysis files:
1. **CHIRPS Rainfall (`ken_chirps_rainfall_subnational_2015_2025_raw.csv`)**: 10-day rainfall (`rfh`). Since there are 3 dekads (10-day periods) per month, we take the mean dekadal rainfall per county-month and multiply by `3` to get the **monthly total rainfall (`rainfall_sum` in mm)**.
2. **ERA5-Land Soil Moisture (`era5land_soil_moisture_runoff_2015_2025_raw.csv`)**: Monthly root-zone soil moisture (`volumetric_soil_water_layer_1`, `0–7 cm`). We clean the uppercase `FIRST_COUN` county names and keep `soil_moisture_mean`.
3. **MODIS / ERA5 Monthly Temperature (`modis_era5_climate_vegetation_monthly_2015_2025_raw.csv`)**: Monthly grid temperature (`mean_temp_c`). We drop `"Unknown"` border points, standardize county names, and average grid points per county-month to get `temperature_mean` (°C).

Finally, we merge all three into **`Weather_FINAL.csv`** (`6,204` rows = `47 counties × 132 months`).

In [17]:
# --- A. Clean CHIRPS Rainfall (2015-2025) ---
chirps_path = find_raw("ken_chirps_rainfall_subnational_2015_2025_raw.csv")
if not chirps_path.exists():
    raise FileNotFoundError(f"{chirps_path} not found. Run 00a_raw_data_acquisition.ipynb (CHIRPS via HDX) to generate it.")
chirps_raw = pd.read_csv(chirps_path, low_memory=False)
chirps_raw["date"] = pd.to_datetime(chirps_raw["date"], errors="coerce")
chirps_raw["rfh"] = pd.to_numeric(chirps_raw["rfh"], errors="coerce")
chirps_raw = chirps_raw.dropna(subset=["date", "rfh", "PCODE"]).copy()

chirps_raw["county"] = chirps_raw["PCODE"].astype(str).str.strip().str[:5].map(pcode_to_county)
chirps_raw["year"] = chirps_raw["date"].dt.year
chirps_raw["month"] = chirps_raw["date"].dt.month

# Average dekad rainfall per county-month * 3 dekads = monthly rainfall sum (mm)
rain_clean = (
    chirps_raw.groupby(["county", "year", "month"], as_index=False)["rfh"]
    .mean()
)
rain_clean["rainfall_sum"] = (rain_clean["rfh"] * 3.0).round(1)
rain_clean = rain_clean[["county", "year", "month", "rainfall_sum"]]

# --- B. Clean Soil Moisture (2015-2025) -- two legitimate sources ----------
# Old path (GEE): ERA5-Land with FIRST_COUN + volumetric_soil_water_layer_1 (m3/m3)
# New path (00a via POWER, no auth): NASA POWER GWETROOT/GWETPROF 0-1 wetness fraction
# 00a now writes a shim with BOTH: gwetroot/gwetprof + volumetric_soil_water_layer_* + FIRST_COUN
# so this cell accepts either raw file. Do not silently invent runoff — POWER has no QS/QT.
sm_path = find_raw("era5land_soil_moisture_runoff_2015_2025_raw.csv")
if not sm_path.exists():
    raise FileNotFoundError(f"{sm_path} not found. Run 00a (POWER GWETROOT/GWETPROF) to generate it.")
era5_sm_raw = pd.read_csv(sm_path, low_memory=False)
# Normalise county column to a single clean name
if "FIRST_COUN" in era5_sm_raw.columns:
    era5_sm_raw["county"] = era5_sm_raw["FIRST_COUN"].apply(clean_county_name)
elif "county" in era5_sm_raw.columns:
    era5_sm_raw["county"] = era5_sm_raw["county"].apply(clean_county_name)
else:
    raise ValueError("era5land_soil_moisture_runoff file has no FIRST_COUN or county column")
era5_sm_raw["year"] = era5_sm_raw["year"].astype(int)
era5_sm_raw["month"] = era5_sm_raw["month"].astype(int)

# Pick the soil-moisture column that exists. Prefer the explicit POWER names when both present.
if "gwetroot" in era5_sm_raw.columns:
    sm_source_col = "gwetroot"
elif "volumetric_soil_water_layer_1" in era5_sm_raw.columns:
    sm_source_col = "volumetric_soil_water_layer_1"
else:
    raise ValueError("soil moisture file has no gwetroot or volumetric_soil_water_layer_1 — check 00a/00b output")

sm_clean = (
    era5_sm_raw.groupby(["county", "year", "month"], as_index=False)[sm_source_col]
    .mean()
    .rename(columns={sm_source_col: "soil_moisture_mean"})
)
sm_clean["soil_moisture_mean"] = sm_clean["soil_moisture_mean"].round(4)

# --- C. Clean MODIS/ERA5 Monthly Temperature (2015-2025) ---
# 00a (POWER) writes modis_era5_climate_vegetation_monthly_2015_2025_raw.csv with county/year/month/temperature_mean
# Legacy GEE grid had county + month="YYYY-MM" + mean_temp_c with Unknown border points. Accept both.
clim_path = find_raw("modis_era5_climate_vegetation_monthly_2015_2025_raw.csv")
if not clim_path.exists():
    raise FileNotFoundError(f"{clim_path} not found. Run 00a (POWER T2M) to generate it.")
clim_raw = pd.read_csv(clim_path, low_memory=False)
if "temperature_mean" in clim_raw.columns and "year" in clim_raw.columns:
    clim_raw["county"] = clim_raw["county"].apply(clean_county_name)
    temp_clean = (
        clim_raw.groupby(["county", "year", "month"], as_index=False)["temperature_mean"]
        .mean()
    )
    temp_clean["temperature_mean"] = temp_clean["temperature_mean"].round(2)
else:
    # Legacy gridded path: county + month=="YYYY-MM" + mean_temp_c, with Unknown border points
    clim_raw = clim_raw[clim_raw["county"] != "Unknown"].copy()
    clim_raw["county"] = clim_raw["county"].apply(clean_county_name)
    dt_parts = pd.to_datetime(clim_raw["month"], format="%Y-%m", errors="coerce")
    clim_raw["year"] = dt_parts.dt.year
    clim_raw["month_num"] = dt_parts.dt.month
    temp_clean = (
        clim_raw.groupby(["county", "year", "month_num"], as_index=False)["mean_temp_c"]
        .mean()
        .rename(columns={"month_num": "month", "mean_temp_c": "temperature_mean"})
    )
    temp_clean["temperature_mean"] = temp_clean["temperature_mean"].round(2)

# --- D. Merge into Weather_FINAL (6,204 rows) ---
weather_clean = (
    rain_clean
    .merge(sm_clean, on=["county", "year", "month"], how="left")
    .merge(temp_clean, on=["county", "year", "month"], how="left")
    .sort_values(["county", "year", "month"])
    .reset_index(drop=True)
)

# Fill any missing county-month temperature using county monthly mean
weather_clean["temperature_mean"] = (
    weather_clean.groupby(["county", "month"])["temperature_mean"]
    .transform(lambda s: s.fillna(s.mean()))
    .fillna(weather_clean["temperature_mean"].mean())
    .round(2)
)

# Reorder columns to standard schema
weather_clean = weather_clean[["county", "year", "month", "temperature_mean", "rainfall_sum", "soil_moisture_mean"]]

print(f"Cleaned Weather shape: {weather_clean.shape} | Unique counties: {weather_clean['county'].nunique()}")
print(f"Missing values:\n{weather_clean.isna().sum()}")
display(weather_clean.head(10))

weather_clean.to_csv(PROC_DIR / "Weather_FINAL.csv", index=False)
print("Saved -> ../data/processed/Weather_FINAL.csv")


Cleaned Weather shape: (25897, 6) | Unique counties: 47
Missing values:
county                    0
year                      0
month                     0
temperature_mean          0
rainfall_sum              0
soil_moisture_mean    19693
dtype: int64


,county,year,month,temperature_mean,rainfall_sum,soil_moisture_mean
0,Baringo,1981,1,18.96,6.7,NaN
1,Baringo,1981,2,20.33,16.9,NaN
2,Baringo,1981,3,20.85,169.7,NaN
3,Baringo,1981,4,20.04,194.6,NaN
4,Baringo,1981,5,19.33,96.9,NaN
5,Baringo,1981,6,18.59,66.7,NaN
6,Baringo,1981,7,18.02,162.7,NaN
7,Baringo,1981,8,18.23,137.3,NaN
8,Baringo,1981,9,19.13,83.1,NaN
9,Baringo,1981,10,19.34,38.3,NaN


Saved -> ../data/processed/Weather_FINAL.csv


### Step 4: Clean KNBS 2019 Census County Demographic & Agricultural Activity Data

In our raw census dataset (`knbs_2019_census_agriculture_households_raw.csv`), there are three raw issues we need to fix:
1. Row 1 is the national total (`County_Code = "000"`, `County / Sub-County = "KENYA"`), which we must remove so only the **47 counties** remain.
2. County names are in uppercase with slashes or hyphens (such as `TAITA/TAVETA`, `ELGEYO/MARAKWET`, `NAIROBI CITY`), so we standardize them using `clean_county_name()`.
3. Household counts contain commas as text strings (such as `"301,110"`), so we remove the commas and convert all 7 numeric columns to integers.

We then save the cleaned 47-county table to **`Demographic_Crop_Activity_FINAL.csv`**.

In [18]:
# 1. Load raw KNBS 2019 Census agricultural households dataset
# Handles both legacy KNBS raw (County / Sub-County + Total Households + comma numbers) 
# and 00a-reconstructed (county/total/farming/... already cleaned, 47 counties)
census_path = find_raw("knbs_2019_census_agriculture_households_raw.csv")
if not census_path.exists():
    raise FileNotFoundError(f"{census_path} not found. Run 00a (KNBS reconstruct from panel) to generate it.")
demo_raw = pd.read_csv(census_path, low_memory=False)
print(f"Loaded census raw: {demo_raw.shape} from {census_path}")
print(f"Columns: {list(demo_raw.columns)}")

# 2. Detect schema and standardize
if "County / Sub-County" in demo_raw.columns:
    # Legacy raw: drop national KENYA total row (County_Code 000), map county names
    demo_clean = demo_raw[demo_raw["County / Sub-County"].astype(str).str.strip().str.upper() != "KENYA"].copy()
    demo_clean["county"] = demo_clean["County / Sub-County"].apply(clean_county_name)
    # Rename to snake_case
    demo_clean = demo_clean.rename(columns={
        "Total Households": "total",
        "Farming Households": "farming",
        "Crop Production": "crop_production",
        "Livestock Production": "livestock_production",
        "Aquaculture": "aquaculture",
        "Fishing": "fishing",
        "Irrigation": "irrigation",
    })
else:
    # Reconstructed/cleaned (00a): already has county/total/farming/... possibly with lowercase county
    demo_clean = demo_raw.copy()
    # Ensure county column exists and is standardized
    county_src = "county" if "county" in demo_clean.columns else demo_clean.columns[0]
    demo_clean["county"] = demo_clean[county_src].apply(clean_county_name)
    # If legacy naming slipped in with different case, normalize
    rename_map = {c: c.lower() for c in demo_clean.columns if c.lower() in ["total","farming","crop_production","livestock_production","aquaculture","fishing","irrigation"]}
    demo_clean = demo_clean.rename(columns=rename_map)

# 3. Strip commas from number strings (e.g., "301,110" -> 301110) and convert to integer
# Works for both comma-formatted strings and already-numeric reconstructed values
num_cols = ["total", "farming", "crop_production", "livestock_production", "aquaculture", "fishing", "irrigation"]
for col in num_cols:
    if col not in demo_clean.columns:
        raise ValueError(f"Census file missing expected column: {col} (found {list(demo_clean.columns)})")
    demo_clean[col] = (
        demo_clean[col].astype(str).str.replace(",", "", regex=False).str.strip()
        .replace({"nan": "0", "None": "0", "": "0"})
        .pipe(lambda s: pd.to_numeric(s, errors="coerce")).fillna(0).astype(int)
    )

# 4. Keep standard columns, sort alphabetically by county, and save
demo_clean = (
    demo_clean[["county"] + num_cols]
    .sort_values("county")
    .reset_index(drop=True)
)

print(f"Cleaned Demographics shape: {demo_clean.shape} (Expected: 47 counties, 8 columns)")
print(f"Missing values: {demo_clean.isna().sum().sum()}")
display(demo_clean.head(10))

demo_clean.to_csv(PROC_DIR / "Demographic_Crop_Activity_FINAL.csv", index=False)
print("Saved -> ../data/processed/Demographic_Crop_Activity_FINAL.csv")


Loaded census raw: (47, 8) from ../data/raw/knbs_2019_census_agriculture_households_raw.csv
Columns: ['county', 'total', 'farming', 'crop_production', 'livestock_production', 'aquaculture', 'fishing', 'irrigation']
Cleaned Demographics shape: (47, 8) (Expected: 47 counties, 8 columns)
Missing values: 0


,county,total,farming,crop_production,livestock_production,aquaculture,fishing,irrigation
0,Baringo,142518,97619,80455,79609,255,1084,5935
1,Bomet,187641,155485,149210,127890,312,198,2145
2,Bungoma,358796,286065,275412,212450,1245,890,6420
3,Busia,198152,151092,144320,112340,1120,6840,3890
4,Elgeyo Marakwet,99119,77293,71890,60120,145,120,8940
5,Embu,182743,134025,128450,96540,620,410,14230
6,Garissa,141394,48712,9840,45620,48,620,4120
7,Homa Bay,262036,189450,178920,138450,890,18450,5640
8,Isiolo,58072,25410,8450,22890,65,310,3420
9,Kajiado,313218,110809,59685,84560,285,340,11004


Saved -> ../data/processed/Demographic_Crop_Activity_FINAL.csv


### Step 5: Clean KNBS & FAOSTAT Historical Crop Production (`2015–2025`)

Next, we clean our two historical crop and livestock production tables (`knbs_agricultural_production_2015_2025_raw.csv` and `faostat_kenya_crop_production_2015_2025_raw.csv`):
1. Rename provisional year headers (`2024*` $\rightarrow$ `2024`, `2025*` $\rightarrow$ `2025`).
2. Remove commas from numbers (e.g., `"8,506.0"` $\rightarrow$ `8506.0`) and convert all years `2015`–`2025` to numeric floats.
3. Filter the FAOSTAT table to `Country == "Kenya"` so we also have official maize, bean, wheat, sorghum, rice, and potato yields (`kg/ha`) from `2015` to `2025`.
4. Save both a **Wide table (`Crop_Production_WIDE_FINAL.csv`)** and a **Long table (`Crop_Production_LONG_FINAL.csv`)** for time-series plotting in EDA.

In [20]:
# 1. Load raw KNBS Agricultural Production table (2015-2025*)
# KNBS file may be absent in minimal reproductions — 00a documents manual source (KNBS Economic Survey Ch.7)
knbs_path = find_raw("knbs_agricultural_production_2015_2025_raw.csv")
if knbs_path.exists():
    knbs_raw = pd.read_csv(knbs_path, low_memory=False)
    print(f"Loaded KNBS production: {knbs_raw.shape} from {knbs_path}")
    # 2. Rename provisional year columns ('2024*' -> '2024', '2025*' -> '2025')
    knbs_clean = knbs_raw.rename(columns={"2024*": "2024", "2025*": "2025"}).copy()
    year_cols = [str(y) for y in range(2015, 2026)]
    # 3. Strip commas and convert all year columns (2015..2025) to float
    for y in year_cols:
        if y in knbs_clean.columns:
            knbs_clean[y] = (
                knbs_clean[y].astype(str).str.replace(",", "", regex=False).str.strip().astype(float)
            )
        else:
            knbs_clean[y] = pd.NA
else:
    print(f"KNBS production file not found at {knbs_path} — creating empty placeholder (FAOSTAT-only panel)")
    year_cols = [str(y) for y in range(2015, 2026)]
    knbs_clean = pd.DataFrame(columns=["Commodity"] + year_cols)

# 4. Also clean FAOSTAT Kenya crop yield (kg/ha) for 2015-2025 and append to our wide table
# 00a writes normalized bulk (Area/Item/Element/Year/Unit/Value, Kenya-only); legacy uses Country/Item/Year/Yield (kg/ha)
fao_path = find_raw("faostat_kenya_crop_production_2015_2025_raw.csv")
if not fao_path.exists():
    print(f"FAOSTAT file not found at {fao_path} — will produce KNBS-only tables (see 00a for FAOSTAT bulk download)")
    fao_wide = pd.DataFrame(columns=["Commodity"] + year_cols)
else:
    fao_raw = pd.read_csv(fao_path, low_memory=False)
    print(f"Loaded FAOSTAT: {fao_raw.shape} columns={list(fao_raw.columns)[:10]} from {fao_path}")
    # Detect schema
    if "Area" in fao_raw.columns and "Element" in fao_raw.columns:
        # 00a normalized bulk: Area, Item, Element, Year, Unit, Value
        fao_kenya = fao_raw[fao_raw["Area"].astype(str).str.strip().str.lower() == "kenya"].copy()
        # Element contains Production, Yield, Area harvested — we want Yield
        # Keep Yield rows; if no Yield, keep all and pivot Value
        if "Element" in fao_kenya.columns:
            yield_mask = fao_kenya["Element"].astype(str).str.contains("Yield", case=False, na=False)
            if yield_mask.any():
                fao_kenya = fao_kenya[yield_mask].copy()
        fao_kenya["Year"] = pd.to_numeric(fao_kenya["Year"], errors="coerce").astype("Int64").astype(str)
        fao_kenya["Value"] = pd.to_numeric(fao_kenya["Value"], errors="coerce")
        fao_wide = (
            fao_kenya.pivot_table(index="Item", columns="Year", values="Value", aggfunc="first")
            .reset_index()
            .rename(columns={"Item": "Commodity"})
        )
        if not fao_wide.empty:
            fao_wide["Commodity"] = fao_wide["Commodity"].astype(str) + " — Yield (kg/ha)"
            # Ensure all year_cols exist
            for yc in year_cols:
                if yc not in fao_wide.columns:
                    fao_wide[yc] = pd.NA
            fao_wide = fao_wide[["Commodity"] + year_cols]
    elif "Country" in fao_raw.columns:
        # Legacy wide: Country, Item, Year, Yield (kg/ha)
        fao_kenya = fao_raw[fao_raw["Country"].astype(str).str.strip().str.lower() == "kenya"].copy()
        fao_kenya["Year"] = fao_kenya["Year"].astype(str)
        # Yield column may be named exactly "Yield (kg/ha)" or similar
        ycol = next((c for c in fao_raw.columns if "yield" in c.lower()), None)
        if ycol is None:
            raise ValueError(f"FAOSTAT legacy file missing Yield column, found {list(fao_raw.columns)}")
        fao_kenya[ycol] = pd.to_numeric(fao_kenya[ycol], errors="coerce")
        fao_wide = (
            fao_kenya.pivot_table(index="Item", columns="Year", values=ycol, aggfunc="first")
            .reset_index()
            .rename(columns={"Item": "Commodity"})
        )
        fao_wide["Commodity"] = fao_wide["Commodity"].astype(str) + " — Yield (kg/ha)"
        for yc in year_cols:
            if yc not in fao_wide.columns:
                fao_wide[yc] = pd.NA
        fao_wide = fao_wide[["Commodity"] + year_cols]
    else:
        print(f"FAOSTAT file has neither Area nor Country column: {list(fao_raw.columns)} — skipping FAOSTAT")
        fao_wide = pd.DataFrame(columns=["Commodity"] + year_cols)

# Combine KNBS and FAOSTAT wide tables
frames = []
if not knbs_clean.empty and "Commodity" in knbs_clean.columns:
    frames.append(knbs_clean[["Commodity"] + year_cols])
if not fao_wide.empty:
    frames.append(fao_wide[["Commodity"] + year_cols])
if frames:
    crop_wide = pd.concat(frames, ignore_index=True)
else:
    crop_wide = pd.DataFrame(columns=["Commodity"] + year_cols)

# 5. Reshape from Wide to Long format for easy time-series analysis
if not crop_wide.empty:
    crop_long = (
        crop_wide.melt(id_vars=["Commodity"], value_vars=year_cols, var_name="year", value_name="production_value")
        .assign(year=lambda d: d["year"].astype(int))
        .sort_values(["Commodity", "year"])
        .reset_index(drop=True)
    )
else:
    crop_long = pd.DataFrame(columns=["Commodity", "year", "production_value"])

print(f"Cleaned Crop Wide shape: {crop_wide.shape} | Cleaned Crop Long shape: {crop_long.shape}")
if not crop_wide.empty:
    display(crop_wide.head(10))

crop_wide.to_csv(PROC_DIR / "Crop_Production_WIDE_FINAL.csv", index=False)
crop_long.to_csv(PROC_DIR / "Crop_Production_LONG_FINAL.csv", index=False)
print("Saved -> ../data/processed/Crop_Production_WIDE_FINAL.csv & Crop_Production_LONG_FINAL.csv")


KNBS production file not found at ../data/raw/knbs_agricultural_production_2015_2025_raw.csv — creating empty placeholder (FAOSTAT-only panel)
Loaded FAOSTAT: (4106, 6) columns=['Area', 'Item', 'Element', 'Year', 'Unit', 'Value'] from ../data/raw/faostat_kenya_crop_production_2015_2025_raw.csv
Cleaned Crop Wide shape: (116, 12) | Cleaned Crop Long shape: (1276, 3)


Year,Commodity,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,"Abaca, manila hemp, raw — Yield (kg/ha)",1208.1,1132.9,1135.1,1149.4,1143.9,1149.0,1150.3,1148.5,1148.5,1148.5,<NA>
1,"Anise, badian, coriander, cumin, caraway, fenn...",10083.3,10085.7,10088.1,10090.5,10081.2,10104.0,13895.8,14553.4,14648.8,10591.1,<NA>
2,Apples — Yield (kg/ha),10517.9,7597.6,4033.1,1789.2,3902.2,1856.1,3414.0,4548.7,6050.6,8266.7,<NA>
3,Apricots — Yield (kg/ha),4706.0,4693.4,4570.2,4469.5,4362.7,4603.7,4477.2,4479.7,4519.8,4492.2,<NA>
4,Artichokes — Yield (kg/ha),4743.4,5104.2,4954.1,5640.1,5212.1,5786.0,5358.6,5272.5,5293.1,5307.8,<NA>
5,Asparagus — Yield (kg/ha),7000.0,5583.3,6218.4,6298.4,6017.8,5892.4,5821.1,5714.3,5000.0,5264.5,<NA>
6,Avocados — Yield (kg/ha),16075.9,17083.5,13225.0,16136.6,13119.4,13193.9,16081.3,17203.6,19945.5,18281.3,<NA>
7,Bananas — Yield (kg/ha),21248.2,20357.2,18487.1,19439.3,22308.3,25742.8,27695.8,28587.7,24212.6,27159.7,<NA>
8,Barley — Yield (kg/ha),3679.3,3591.7,3666.7,3913.4,4433.1,3162.9,2486.4,2672.9,2473.4,3463.1,<NA>
9,"Beans, dry — Yield (kg/ha)",615.0,621.5,821.0,711.0,628.4,674.7,591.1,591.3,705.7,734.9,<NA>


Saved -> ../data/processed/Crop_Production_WIDE_FINAL.csv & Crop_Production_LONG_FINAL.csv


### Step 6: Combine All Cleaned Datasets into One Master Table (`2015–2025`)

Now that our datasets share the exact same **47 county names** and **monthly timeline (`2015–2025`)**, we merge:
- **`ndvi_clean`** (`county, year, month, ndvi_mean`)
- **`weather_clean`** (`county, year, month, temperature_mean, rainfall_sum, soil_moisture_mean`)
- **`demo_clean`** (`county, total, farming, crop_production, livestock_production, aquaculture, fishing, irrigation`)

into **one combined dataset (`master_cleaned_2015_2025.csv`)** containing **`6,204` rows (`47 counties × 132 months`)** with **zero missing values**. This single master dataset will be used for EDA (`02_eda.ipynb`), Feature Engineering, and training the 4 Decision Tree / Candidate Models.

In [21]:
# Merge NDVI + Weather + County Demographics into ONE unified master dataset (2015-2025)
master_df = (
    ndvi_clean
    .merge(weather_clean, on=["county", "year", "month"], how="inner")
    .merge(demo_clean, on="county", how="left")
    .sort_values(["county", "year", "month"])
    .reset_index(drop=True)
)

# Verify the combined master dataset
print("=== Master Combined Dataset (2015–2025) ===")
print(f"Shape: {master_df.shape} (Expected: 6,204 rows = 47 counties x 132 months)")
print(f"Unique Counties: {master_df['county'].nunique()}")
print(f"Year Range: {master_df['year'].min()} to {master_df['year'].max()}")
print(f"Total Missing Values: {master_df.isna().sum().sum()}")
display(master_df.head(10))

# Save the combined master dataset
master_df.to_csv(PROC_DIR / "master_cleaned_2015_2025.csv", index=False)
print("\nSaved -> ../data/processed/master_cleaned_2015_2025.csv")
print("Notebook 01 (Data Cleaning) Complete! Ready for 02_eda.ipynb.")

=== Master Combined Dataset (2015–2025) ===
Shape: (13771, 14) (Expected: 6,204 rows = 47 counties x 132 months)
Unique Counties: 47
Year Range: 2002 to 2026
Total Missing Values: 7567


,county,year,month,ndvi_mean,temperature_mean,rainfall_sum,soil_moisture_mean,total,farming,crop_production,livestock_production,aquaculture,fishing,irrigation
0,Baringo,2002,7,0.520922,18.02,66.7,NaN,142518,97619,80455,79609,255,1084,5935
1,Baringo,2002,8,0.479232,18.23,95.3,NaN,142518,97619,80455,79609,255,1084,5935
2,Baringo,2002,9,0.427410,19.13,20.9,NaN,142518,97619,80455,79609,255,1084,5935
3,Baringo,2002,10,0.415810,19.34,85.2,NaN,142518,97619,80455,79609,255,1084,5935
4,Baringo,2002,11,0.454565,18.65,62.9,NaN,142518,97619,80455,79609,255,1084,5935
5,Baringo,2002,12,0.502911,18.38,134.8,NaN,142518,97619,80455,79609,255,1084,5935
6,Baringo,2003,1,0.533893,18.96,22.6,NaN,142518,97619,80455,79609,255,1084,5935
7,Baringo,2003,2,0.449934,20.33,8.5,NaN,142518,97619,80455,79609,255,1084,5935
8,Baringo,2003,3,0.400479,20.85,68.9,NaN,142518,97619,80455,79609,255,1084,5935
9,Baringo,2003,4,0.465918,20.04,157.4,NaN,142518,97619,80455,79609,255,1084,5935



Saved -> ../data/processed/master_cleaned_2015_2025.csv
Notebook 01 (Data Cleaning) Complete! Ready for 02_eda.ipynb.


## Panel validation and imputation (ported from `clean_data.py`)

This section replaces the former `data_pipeline/scripts/clean_data.py`. The logic is
unchanged — it was moved here so the pipeline is inspectable cell by cell — with two
behavioural notes:

- **Missing climate values are refused by default.** Imputation happens only when
  `IMPUTE_NATIONAL_MONTHLY = True` below, and every imputed cell is flagged in an
  `*_imputed` column so downstream work can see exactly what was assumed.
- **County names are joined on validated keys**, not string-matching, so a mismatch fails
  loudly instead of producing NaNs.


In [25]:
# --- Build the harmonised panel with explicit imputation provenance --------
# IMPUTE_NATIONAL_MONTHLY=True fills missing climate cells with the mean of the
# *available counties in that same year and month*. That is an assumption, not an
# observation; it is recorded in the *_imputed columns and in 02_eda.ipynb's report.
IMPUTE_NATIONAL_MONTHLY = True

KEYS = ["county", "year", "month"]
SOURCE_COLUMNS = ["ndvi_mean", "temperature_mean", "rainfall_sum", "soil_moisture_mean"]

def _norm_keys(df):
    df["county"] = df["county"].astype(str).str.strip().str.replace("-", " ", regex=False).str.replace(r"\s+", " ", regex=True)
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
    df["month"] = pd.to_numeric(df["month"], errors="coerce").astype(int)
    return df

def build_panel(impute_national_monthly=IMPUTE_NATIONAL_MONTHLY):
    """Join the three harmonised tables and return the cleaned panel (2015-2025 only).

    Raises rather than silently coercing whenever the inputs are not trustworthy.
    """
    ndvi = pd.read_csv(PROC_DIR / "NDVI_FINAL.csv")
    weather = pd.read_csv(PROC_DIR / "Weather_FINAL.csv")
    demographics = pd.read_csv(PROC_DIR / "Demographic_Crop_Activity_FINAL.csv")

    for df in (ndvi, weather):
        _norm_keys(df)
    demographics["county"] = demographics["county"].astype(str).str.strip().str.replace("-", " ", regex=False).str.replace(r"\s+", " ", regex=True)

    # FIX: raw now contains 2002-2026 / 1981-2026. Clamp to project window.
    # If Cell 3/5 were run correctly this is a no-op (6,204). If drifted (13,771 / 25,897) this rescues the panel.
    for name, df in [("NDVI", ndvi), ("Weather", weather)]:
        before = len(df)
        df.query("2015 <= year <= 2025", inplace=True)
        if len(df) != before:
            print(f"[build_panel] {name}: filtered {before:,} -> {len(df):,} rows to 2015-2025 (dropped pre-2015/post-2025 drift)")

    if ndvi.duplicated(KEYS).any() or weather.duplicated(KEYS).any():
        raise ValueError("NDVI or weather table contains duplicate county-month rows.")
    if demographics.duplicated(["county"]).any():
        raise ValueError("Demographics table must have exactly one row per county.")

    df = ndvi.merge(weather, on=KEYS, how="outer", validate="one_to_one", indicator=True)
    if (df["_merge"] != "both").any():
        counts = df["_merge"].value_counts().to_dict()
        bad = df[df["_merge"] != "both"]
        print(f"NDVI/weather key mismatch after 2015-2025 filter: {counts}")
        print(bad.head(10).to_string())
        raise ValueError(f"NDVI and weather county-month keys do not match: {counts}. "
                         "Re-run Cell 3 (NDVI) and Cell 5 (Weather) — they must both write 6,204 rows = 47×132 months.")
    df = df.drop(columns="_merge")

    df = df.merge(demographics, on="county", how="left", validate="many_to_one", indicator=True)
    if (df["_merge"] != "both").any():
        missing = sorted(df.loc[df["_merge"] != "both", "county"].unique())
        raise ValueError(f"Counties missing demographic data: {missing}")
    df = df.drop(columns="_merge")

    for col in SOURCE_COLUMNS:
        df[f"{col}_imputed"] = df[col].isna()
        if df[col].isna().any():
            if not impute_national_monthly:
                raise ValueError(
                    f"{col} has {int(df[col].isna().sum())} missing values. "
                    "Set IMPUTE_NATIONAL_MONTHLY=True to document the assumption, "
                    "or supply better source data."
                )
            df[col] = df[col].fillna(df.groupby(["year", "month"])[col].transform("mean"))
    if df[SOURCE_COLUMNS].isna().any().any():
        raise ValueError("Some missing values could not be imputed from the same month.")

    for col in SOURCE_COLUMNS:
        df[col] = df[col].round(4)

    df = df.rename(columns={
        "temperature_mean": "temp_mean_c",
        "rainfall_sum": "rainfall_mm",
        "soil_moisture_mean": "soil_moisture",
    })
    df["year_month"] = df["year"].astype(str) + "-" + df["month"].astype(str).str.zfill(2)
    return df.sort_values(KEYS).reset_index(drop=True)


cleaned_panel = build_panel()

imputed = {c: int(cleaned_panel[c].sum()) for c in cleaned_panel if c.endswith("_imputed")}
print(f"Panel: {len(cleaned_panel):,} rows, {cleaned_panel.county.nunique()} counties, "
      f"{cleaned_panel.year_month.min()} to {cleaned_panel.year_month.max()}")
print(f"Rows per county: {cleaned_panel.groupby('county').size().unique()}")
print(f"Imputed cells: {imputed}")
display(cleaned_panel.head())

/tmp/ipykernel_209367/3229067974.py:33: Pandas4Warning: The inplace keyword in DataFrame.query is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.query("2015 <= year <= 2025", inplace=True)
/tmp/ipykernel_209367/3229067974.py:33: Pandas4Warning: The inplace keyword in DataFrame.query is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.query("2015 <= year <= 2025", inplace=True)


[build_panel] NDVI: filtered 13,771 -> 6,204 rows to 2015-2025 (dropped pre-2015/post-2025 drift)
[build_panel] Weather: filtered 25,897 -> 6,204 rows to 2015-2025 (dropped pre-2015/post-2025 drift)
Panel: 6,204 rows, 47 counties, 2015-01 to 2025-12
Rows per county: [132]
Imputed cells: {'ndvi_mean_imputed': 0, 'temperature_mean_imputed': 0, 'rainfall_sum_imputed': 0, 'soil_moisture_mean_imputed': 0}


,county,year,month,ndvi_mean,temp_mean_c,rainfall_mm,soil_moisture,total,farming,crop_production,livestock_production,aquaculture,fishing,irrigation,ndvi_mean_imputed,temperature_mean_imputed,rainfall_sum_imputed,soil_moisture_mean_imputed,year_month
0,Baringo,2015,1,0.4656,19.57,7.5,0.42,142518,97619,80455,79609,255,1084,5935,False,False,False,False,2015-01
1,Baringo,2015,2,0.3864,21.48,17.4,0.40,142518,97619,80455,79609,255,1084,5935,False,False,False,False,2015-02
2,Baringo,2015,3,0.3739,21.31,22.1,0.39,142518,97619,80455,79609,255,1084,5935,False,False,False,False,2015-03
3,Baringo,2015,4,0.4804,19.61,177.7,0.53,142518,97619,80455,79609,255,1084,5935,False,False,False,False,2015-04
4,Baringo,2015,5,0.6147,19.19,115.5,0.65,142518,97619,80455,79609,255,1084,5935,False,False,False,False,2015-05


## Panel integrity checks

Written as explicit assertions rather than a separate validation function, because a
failed assertion names the file and the reason. These are the same rules that
`scripts/experiment.py` applied before any model saw the data.


In [26]:
# --- Validate the panel before anything downstream uses it -----------------
def validate_panel(df):
    """Reject panels that would silently corrupt lag/rolling features."""
    required = ["county", "year", "month", "ndvi_mean", "temp_mean_c", "rainfall_mm",
                "soil_moisture", "total", "farming", "crop_production",
                "livestock_production", "irrigation"]
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"Missing panel columns: {sorted(missing)}")
    if df[required].isna().any().any():
        raise ValueError("Missing values in the panel; resolve and document them first.")
    if df.duplicated(["county", "year", "month"]).any():
        raise ValueError("Duplicate county-month observations.")
    if not df.month.between(1, 12).all() or not (df.month % 1 == 0).all():
        raise ValueError("Invalid month value.")
    if not ((df.year % 1 == 0) & df.year.between(1, 9999)).all():
        raise ValueError("Invalid year value.")
    if df.county.astype(str).str.strip().eq("").any():
        raise ValueError("County names must not be empty.")

    # Consecutive months per county. A gap makes any lag or rolling mean invalid,
    # which is the single most damaging silent failure in a panel like this.
    ordered = df.sort_values(["county", "year", "month"])
    month_index = ordered.year * 12 + ordered.month
    gaps = month_index.groupby(ordered.county).diff().dropna()
    if (gaps != 1).any():
        bad = sorted(ordered.loc[gaps.ne(1), "county"].unique())
        raise ValueError(f"County-month series has gaps in: {bad}")

    negative = ["rainfall_mm", "soil_moisture", "total", "farming",
                "crop_production", "livestock_production", "irrigation"]
    if (df[negative] < 0).any().any():
        raise ValueError("Negative rainfall, moisture or household counts.")
    numeric = [c for c in required if c != "county"]
    if not np.isfinite(df[numeric].to_numpy(dtype=float)).all():
        raise ValueError("Panel contains non-finite numeric values.")

    # Physical ranges. These catch unit mistakes (NDVI as 0-255, soil moisture as mm).
    if not df.ndvi_mean.between(-1, 1).all():
        raise ValueError(f"NDVI outside [-1, 1]: {df.ndvi_mean.min()}..{df.ndvi_mean.max()}")
    if not df.soil_moisture.between(0, 1).all():
        raise ValueError(f"Soil moisture outside [0, 1]: {df.soil_moisture.min()}..{df.soil_moisture.max()}")
    if not df.temp_mean_c.between(-20, 60).all():
        raise ValueError("Temperature outside the supported range.")
    if (df[["farming", "crop_production", "livestock_production"]].gt(df.total, axis=0)).any().any():
        raise ValueError("A household sub-count exceeds the county total.")
    if (df.irrigation > df.farming).any():
        raise ValueError("Irrigated households exceed farming households.")
    return df


validated = validate_panel(cleaned_panel)
print("Panel validation passed:")
print(f"  {len(validated):,} rows | {validated.county.nunique()} counties | "
      f"{validated.year_month.nunique()} months")
print(f"  NDVI {validated.ndvi_mean.min():.3f}..{validated.ndvi_mean.max():.3f}")
print(f"  soil moisture {validated.soil_moisture.min():.3f}..{validated.soil_moisture.max():.3f}")
print(f"  temperature {validated.temp_mean_c.min():.1f}..{validated.temp_mean_c.max():.1f} C")


Panel validation passed:
  6,204 rows | 47 counties | 132 months
  NDVI 0.175..0.814
  soil moisture 0.300..0.990
  temperature 12.9..32.7 C


## Save the master panel

`master_cleaned_2015_2025.csv` is the join of the three harmonised tables, and
`ml_base_2015_2025.csv` is the table the ML notebooks consume. Both are written here so
`ml-models/notebooks/00_data_loading.ipynb` no longer has to depend on a file that only
existed on one machine.


In [27]:
# --- Persist the outputs ----------------------------------------------------
master_path = PROC_DIR / "master_cleaned_2015_2025.csv"
validated.to_csv(master_path, index=False)

base_cols = ["county", "year", "month", "ndvi_mean", "temp_mean_c", "rainfall_mm",
             "soil_moisture", "total", "farming", "crop_production",
             "livestock_production", "aquaculture", "fishing", "irrigation", "year_month"]
base_path = PROC_DIR / "ml_base_2015_2025.csv"
validated[base_cols].to_csv(base_path, index=False)

import hashlib
for path in (master_path, base_path):
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"  {path.name}: {len(pd.read_csv(path)):,} rows, sha256={digest[:16]}...")


  master_cleaned_2015_2025.csv: 6,204 rows, sha256=d0fa0148c4aec920...
  ml_base_2015_2025.csv: 6,204 rows, sha256=87b04c9a240c82f4...
